# Lie Groups — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Lie Groups exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/lie_companion.ipynb)

## Setup

In [31]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [32]:
import numpy as np
from numpy import einsum as ein
from typing import List

from symm4ml import groups, linalg, rep, lie

### Reference data

These structure constants and generators are used throughout the exercise for testing.

In [33]:
# SO(3) structure constants
so3_A = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) L=1 generators
so3_X = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(1,3) structure constants
so13_A = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
        ],
        [
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SO(1,3) generators
so13_X = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 1.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SU(2) generators
su2_X = np.array(
    [
        [
            [0.0 + 0.0j, 0.5 + 0.0j],
            [-0.5 + 0.0j, 0.0 + 0.0j],
        ],
        [
            [-0.0 - 0.5j, 0.0 + 0.0j],
            [0.0 + 0.0j, 0.0 + 0.5j],
        ],
        [
            [0.0 - 0.0j, 0.0 + 0.5j],
            [0.0 + 0.5j, 0.0 - 0.0j],
        ],
    ]
)

print(f"so3_A: {so3_A.shape}, so3_X: {so3_X.shape}")
print(f"so13_A: {so13_A.shape}, so13_X: {so13_X.shape}")
print(f"su2_X: {su2_X.shape}")

so3_A: (3, 3, 3), so3_X: (3, 3, 3)
so13_A: (6, 6, 6), so13_X: (6, 4, 4)
su2_X: (3, 2, 2)


---
## 1. `are_isomorphic(X1, X2)`

Check if two representations of a Lie algebra (expressed in terms of generators) are isomorphic, i.e., the same up to a similarity transform. You can use a function from a previous problem set.

$$ Ue^{X_1} = e^{X_2}U$$
$$ Ue^X U^{-1} = e^{UXU^{-1}} $$
$$ Ue^{X_1}U^{-1} = e^{X_2} $$
$$ e^{UX_1U^{-1}} = e^{X_2}

In [34]:
def are_isomorphic(X1: np.ndarray, X2: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if two representations of a Lie group are isomorphic.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        are_isomorphic: bool - True if the representations are isomorphic.
    """
    return rep.are_isomorphic(X1, X2, tol=tol)

In [35]:
# Small tests from lie.py
assert are_isomorphic(so3_X, so3_X)
print("are_isomorphic tests passed!")

are_isomorphic tests passed!


---
## 2. `tensor_product(X1, X2)`

Compute the tensor product of two representations of a Lie algebra, where both input and output are expressed in terms of generators.

In [36]:
def tensor_product(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    """Tensor product of two representations of a Lie group.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        X: np.array [lie_dim, d1*d2, d1*d2] - tensor product of the representations.
    """
    return np.kron(X1, np.eye(X2.shape[1])) + np.kron(np.eye(X1.shape[1]), X2)

In [37]:
# Small tests from lie.py
assert tensor_product(so3_X, so3_X).shape == (3, 9, 9)
print("tensor_product tests passed!")

tensor_product tests passed!


---
## 3. `is_a_representation(algebra, X)`

Check whether the given generators `X` satisfy the commutation relations encoded in `algebra`: $[X_i, X_j] = \sum_k A_{ijk} X_k$.

In [38]:
def is_a_representation(
    algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8
) -> bool:
    """Check if X satisfies the commutation relations of the Lie algebra:
        [X_i, X_j] = sum_k A_{ijk} X_k
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_a_representation: bool - True if X satisfies the commutation relations.
    """
    lie_dim = algebra.shape[0]
    for i in range(0,lie_dim):
        for j in range(0, lie_dim):
            left = np.matmul(X[i], X[j]) - np.matmul(X[j], X[i])
            alg_vec = algebra[i,j]
            right = np.zeros((X.shape[1], X.shape[2]))
            for k in range(0, lie_dim):
                right = right + alg_vec[k]*X[k]
            if not np.allclose(left, right, rtol=tol):
                return False
    return True

In [39]:
# Small tests from lie.py
assert is_a_representation(so3_A, so3_X)
print("is_a_representation tests passed!")

is_a_representation tests passed!


---
## 4. `is_an_irrep(algebra, X)`

Check if a representation of a Lie algebra is irreducible. Return `True` if the input is a valid representation AND is irreducible.

In [40]:
def is_an_irrep(algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if X is an irreducible representation of the Lie algebra.
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_an_irrep: bool - True if X is an irreducible representation.
    """
    if not is_a_representation(algebra, X, tol=tol):
        return False
    q = linalg.infer_change_of_basis(X, X)
    if q.shape[0] == 1:
        return True
    return False

In [41]:
# Small tests from lie.py
assert is_an_irrep(so3_A, so3_X)
print("is_an_irrep tests passed!")

is_an_irrep tests passed!


---
## 5. `decompose_rep_into_irreps(X)`

Decompose a representation of a Lie algebra into a direct sum of irreducible representations. Follow the same algorithm as for finite groups.

In [42]:
def decompose_rep_into_irreps(X: np.array, *, tol: float = 1e-8) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        Ys: List[np.array] - list of generators of irreducible representations.
    """
    n = len(X)
    # step 1
    # dim of Q is some m rxr square matrices
    q = linalg.infer_change_of_basis(X, X)
    # check if q is just one matrix or not
    if len(q.shape) == 2:
        q = [q]
    m = len(q)
    q_bar = np.zeros((len(q[0]), len(q[0][0])))
    w = []
    # step 2
    # |alpha| = m
    alpha = np.random.rand(m)
    for i in range(0,m):
        q_bar_i = alpha[i] * q[i]
        q_bar = q_bar + q_bar_i
        # step 3: find eval and evect, then find espace W
    q_bar_eval, q_bar_evect = np.linalg.eig(q_bar)
    w = linalg.eigenspaces(q_bar_eval, q_bar_evect, tol=tol)
    l = len(w)
    p = []
    b = []
    for j in range(0,l):
        b_j, p_j = linalg.gram_schmidt(w[j][1].T)
        p.append(p_j)
        b.append(b_j.T)
    rho = []
    for k in range(0,l):
        rho_k = []
        #print(f"b[{k}] = {b[k]}")
        for g in range(0,n):
            rho_k_m = np.matmul(b[k].conj().T, np.matmul(X[g],b[k]))
            rho_k.append(rho_k_m)
        rho.append(np.array(rho_k))
    return rho

In [43]:
# Small tests from lie.py
assert {
    X.shape[1] for X in decompose_rep_into_irreps(tensor_product(so3_X, so3_X))
} == {1, 3, 5}
print("decompose_rep_into_irreps tests passed!")

decompose_rep_into_irreps tests passed!


---
## 6. `infer_irreps_from_tensor_products(X, n)`

Infer `n` non-isomorphic finite-dimensional irreps of the underlying matrix Lie group by successively decomposing tensor products. Start with the trivial representation and take successive tensor products with `X` and (if needed) its conjugate `X*`.

I think the steps should be:
1. Take the input $X$ and get a new value $X'=0 \oplus X \oplus X^*$ 
2. Maybe tensor product $X'$ with itself? unclear
3. Find the irreps with `decompose_rep_into_irreps`, and add it to a list of irreps 
4. Take that irrep, and $\oplus^{kr}$ it with $X'$ to get a new $X'$
5. Do steps 3-4 until the list of irreps is at least len $n$, then return that list

In [70]:
def infer_irreps_from_tensor_products(
    X: np.ndarray, n: int, *, tol: float = 1e-8
) -> List[np.ndarray]:
    """Infers irreducible representations from successive tensor products of a representation.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
        n: int - number of non-isomorphic irreducible representations to infer.
    Output:
        Ys: List[np.array] - list of n generators of irreducible representations,
            each an np.array of shape [lie_dim, d', d'] for some d'.
    """
    x_prime = rep.direct_sum(np.zeros((X.shape)),rep.direct_sum(X, np.conj(X)))
    print(f"x_prime shape = {x_prime.shape}")
    m = 0
    irreps = []
    irr = lie.decompose_rep_into_irreps(x_prime, tol=tol)
    for ir in irr:
        if(not any((lie.are_isomorphic(ir, irrep_, tol=tol) for irrep_ in irreps))):
            #new_irrs.append(ir)
            irreps.append(ir)
            print(f"x_prime shape = {x_prime.shape}, irreps len = {len(irreps)}")
    print(len(irreps))
    while len(irreps) < n+2:
        print(f"entering while loop, tensoring irrep with shape {irreps[m].shape}")
        for k in range(0,m+1):
            print(f"m={m}, k={k}")
            print(f"entering while loop, tensoring irrep with shape {irreps[m].shape} with {irreps[k].shape}")
            new_i_p = lie.tensor_product(irreps[m], irreps[k])
            print(f"new_i_p shape {new_i_p.shape}")
            new_irrep = decompose_rep_into_irreps(new_i_p, tol=tol)
            print(f"{len(new_irrep)} new irreps")
            #print(f"new_irrep shape = {new_irrep[0].shape}")
            for ir in new_irrep:
                print(f"new irrep shape: {ir.shape}")
                if(not any((lie.are_isomorphic(ir, irrep_, tol=tol) for irrep_ in irreps))):
                    #new_irrs.append(ir)
                    irreps.append(ir)
                    if len(irreps) == n+2: break
                    print(f"new irrep shape = {ir.shape},irreps len = {len(irreps)}")
            if len(irreps) == n+2: break
        m = m + 1
    irreps = sorted(irreps, key=lambda x: x.shape[1])
    return irreps[:n]

In [59]:
{(is_an_irrep(so3_A, X), X.shape) for X in infer_irreps_from_tensor_products(so3_X, 4)}

x_prime shape = (3, 9, 9)
x_prime shape = (3, 9, 9), irreps len = 1
x_prime shape = (3, 9, 9), irreps len = 2
2
entering while loop, tensoring irrep with shape (3, 1, 1)
m=0, k=0
entering while loop, tensoring irrep with shape (3, 1, 1) with (3, 1, 1)
new_i_p shape (3, 1, 1)
1 new irreps
new irrep shape: (3, 1, 1)
entering while loop, tensoring irrep with shape (3, 3, 3)
m=1, k=0
entering while loop, tensoring irrep with shape (3, 3, 3) with (3, 1, 1)
new_i_p shape (3, 3, 3)
1 new irreps
new irrep shape: (3, 3, 3)
m=1, k=1
entering while loop, tensoring irrep with shape (3, 3, 3) with (3, 3, 3)
new_i_p shape (3, 9, 9)
3 new irreps
new irrep shape: (3, 5, 5)
new irrep shape = (3, 5, 5),irreps len = 3
new irrep shape: (3, 3, 3)
new irrep shape: (3, 1, 1)
entering while loop, tensoring irrep with shape (3, 5, 5)
m=2, k=0
entering while loop, tensoring irrep with shape (3, 5, 5) with (3, 1, 1)
new_i_p shape (3, 5, 5)
1 new irreps
new irrep shape: (3, 5, 5)
m=2, k=1
entering while loop, ten

{(True, (3, 1, 1)), (True, (3, 3, 3)), (True, (3, 5, 5)), (True, (3, 7, 7))}

In [66]:
t = infer_irreps_from_tensor_products(so13_X, 5)
lie.is_an_irrep(so13_A,t[3])

x_prime shape = (6, 12, 12)
x_prime shape = (6, 12, 12), irreps len = 1
x_prime shape = (6, 12, 12), irreps len = 2
2
entering while loop, tensoring irrep with shape (6, 1, 1)
m=0, k=0
entering while loop, tensoring irrep with shape (6, 1, 1) with (6, 1, 1)
new_i_p shape (6, 1, 1)
1 new irreps
new irrep shape: (6, 1, 1)
entering while loop, tensoring irrep with shape (6, 4, 4)
m=1, k=0
entering while loop, tensoring irrep with shape (6, 4, 4) with (6, 1, 1)
new_i_p shape (6, 4, 4)
1 new irreps
new irrep shape: (6, 4, 4)
m=1, k=1
entering while loop, tensoring irrep with shape (6, 4, 4) with (6, 4, 4)
new_i_p shape (6, 16, 16)
4 new irreps
new irrep shape: (6, 1, 1)
new irrep shape: (6, 3, 3)
new irrep shape = (6, 3, 3),irreps len = 3
new irrep shape: (6, 3, 3)
new irrep shape = (6, 3, 3),irreps len = 4
new irrep shape: (6, 9, 9)
new irrep shape = (6, 9, 9),irreps len = 5
entering while loop, tensoring irrep with shape (6, 3, 3)
m=2, k=0
entering while loop, tensoring irrep with shape (

True

In [68]:
lie.is_an_irrep(so13_A,t[4])

True

In [71]:
for j in infer_irreps_from_tensor_products(so13_X, 5):
    print(f"shape = {j.shape}, irrep? {lie.is_an_irrep(so13_A, j)}")

x_prime shape = (6, 12, 12)
x_prime shape = (6, 12, 12), irreps len = 1
x_prime shape = (6, 12, 12), irreps len = 2
2
entering while loop, tensoring irrep with shape (6, 1, 1)
m=0, k=0
entering while loop, tensoring irrep with shape (6, 1, 1) with (6, 1, 1)
new_i_p shape (6, 1, 1)
1 new irreps
new irrep shape: (6, 1, 1)
entering while loop, tensoring irrep with shape (6, 4, 4)
m=1, k=0
entering while loop, tensoring irrep with shape (6, 4, 4) with (6, 1, 1)
new_i_p shape (6, 4, 4)
1 new irreps
new irrep shape: (6, 4, 4)
m=1, k=1
entering while loop, tensoring irrep with shape (6, 4, 4) with (6, 4, 4)
new_i_p shape (6, 16, 16)
4 new irreps
new irrep shape: (6, 9, 9)
new irrep shape = (6, 9, 9),irreps len = 3
new irrep shape: (6, 1, 1)
new irrep shape: (6, 3, 3)
new irrep shape = (6, 3, 3),irreps len = 4
new irrep shape: (6, 3, 3)
new irrep shape = (6, 3, 3),irreps len = 5
entering while loop, tensoring irrep with shape (6, 9, 9)
m=2, k=0
entering while loop, tensoring irrep with shape (

In [ ]:
# Small tests from lie.py
Xs = infer_irreps_from_tensor_products(so3_X, 3)
print(len(Xs))
assert len(Xs) == 3
assert Xs[0].shape == (3, 1, 1)
assert is_an_irrep(so3_A, Xs[0])
print(Xs[1].shape)
assert Xs[1].shape == (3, 3, 3)
assert is_an_irrep(so3_A, Xs[1])
assert Xs[2].shape == (3, 5, 5)
assert is_an_irrep(so3_A, Xs[2])
print("infer_irreps_from_tensor_products tests passed!")

(3, 6, 6)
x_prime shape = (3, 9, 9)
new_irrep shape = (3, 1, 1)
3
(3, 1, 1)


AssertionError: 

---
## Explore Further

In [ ]:
# Try inferring irreps of SO(1,3) or SU(2)!